In [0]:
%run ./21_parser_tests

In [0]:
import os, importlib, silver_loader
importlib.reload(silver_loader)
from silver_loader import run, RAW_ROOT, CATALOG

# Drop the sample responses into the bronze volume, as if the API had saved them
samples = {
    f"{RAW_ROOT}/day_ahead_price/DE_LU/2025/test_t1.xml":   t1,
    f"{RAW_ROOT}/day_ahead_price/FR/2025/test_t2.xml":      t2,
    f"{RAW_ROOT}/generation_actual/DE_LU/2025/test_t3.xml": t3,
    f"{RAW_ROOT}/physical_flow/FR_ES/2025/test_t5.xml":     t5,
}
for path, body in samples.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(body)

ds = ["day_ahead_price", "generation_actual", "physical_flow"]
print("Run 1 (new files):  ", run(spark, ds))

price = spark.table(f"{CATALOG}.silver.price")
gen   = spark.table(f"{CATALOG}.silver.generation")
flow  = spark.table(f"{CATALOG}.silver.flow")

# Prices: DE_LU hourly with local Berlin time, FR 15-minute averaged to the hour
de = price.filter("zone_code = 'DE_LU'").orderBy("timestamp_utc").collect()
assert len(de) == 4
assert str(de[0].local_date) == "2025-06-02" and de[0].local_hour == 0, "22:00 UTC should be 00:00 Berlin (CEST)"
assert de[2].price_eur_mwh == -5.2
fr = price.filter("zone_code = 'FR'").collect()
assert len(fr) == 1 and abs(fr[0].price_eur_mwh - 10.75) < 1e-9, "Mean of 12, 12, 9.5, 9.5"
assert fr[0].points_in_hour == 4 and fr[0].filled_points == 2 and fr[0].source_resolution == "PT15M"
assert fr[0].local_hour == 12, "10:00 UTC should be 12:00 Paris (CEST)"
print("Prices passed: hourly mean, A03 fill counted, local time correct")

# Generation: solar as generation, pumped storage as consumption
assert gen.count() == 4
assert gen.filter("psr_code = 'B16' AND flow_type = 'generation'").count() == 2
assert gen.filter("psr_code = 'B10' AND flow_type = 'consumption'").count() == 2
print("Generation passed: fuel codes and consumption split")

# Flows: direction mapped to zone codes
fl = flow.collect()
assert len(fl) == 2 and fl[0].from_zone == "FR" and fl[0].to_zone == "ES"
print("Flows passed: FR -> ES")

# Idempotency
print("Run 2 (no new files):", run(spark, ds))
print("Run 3 (reprocess):   ", run(spark, ds, reprocess=True))
assert price.count() == 5 and gen.count() == 4 and flow.count() == 2, "Reprocessing must not duplicate rows"
print("Idempotency passed: re-running updates rows, no duplicates")

print("\nAll silver tests passed.")
display(price.orderBy("zone_code", "timestamp_utc"))

In [0]:
for path in samples:
    if os.path.exists(path):
        os.remove(path)
for t in ["price", "generation", "flow"]:
    spark.sql(f"DELETE FROM {CATALOG}.silver.{t} WHERE source_file LIKE '%/test_%'")
spark.sql(f"DELETE FROM {CATALOG}.ops.silver_file_log WHERE file_path LIKE '%/test_%'")
print("Test data removed. Rows left in silver.price:", spark.table(f"{CATALOG}.silver.price").count())